# Lab 4 — CNNs & Transfer Learning (TensorFlow / Keras)
**Advanced Methods of Artificial Intelligence**

**Goal:** Train and evaluate an image classifier using:
1) a **baseline CNN**, and  
2) **transfer learning** (a pre-trained backbone).

You will also practice **evaluation beyond a single number**: learning curves, confusion matrix, and error analysis.

## Colab-friendly version (memory safe)

This notebook is adapted to run reliably on **Google Colab (≈12 GB RAM)**.
Main changes:
- **No in-memory `dataset.cache()`** by default (this is the most common cause of RAM crashes).
- **Streaming evaluation** (we do *not* store all test images in RAM).
- Smaller recommended defaults (**IMG_SIZE** and **BATCH_SIZE**) and optional **mixed precision** for GPU.

If your dataset is on Google Drive, mount it and set `LOCAL_DATA_DIR` to the correct folder.


## Learning outcomes
By the end of this lab, you can:
- Construct an image pipeline using **`tf.data`**.
- Train a baseline CNN and interpret **training vs validation** behavior.
- Apply transfer learning: **freeze → train head → (optional) fine-tune**.
- Evaluate responsibly: **confusion matrix + misclassified examples**.
- Write a short academic memo: **goal → method → results → limitations → recommendation → risks**.

In [ ]:
# --- Setup & imports (read this once) ---
# Goal: load the libraries we need for vision models + evaluation.
# Tip: Colab has ~12 GB *system RAM*. If you later see a RAM crash, do NOT change this cell.
# Instead change the CONFIG cell (IMG_SIZE, BATCH_SIZE, CACHE_MODE).
#
# Libraries:
# - numpy/pandas: numeric + tables
# - matplotlib: plots (learning curves, confusion matrix)
# - tensorflow/keras: CNN + transfer learning
# - sklearn.metrics: confusion matrix + classification report


# If you run this in Colab, you may need:
# !pip -q install -U tensorflow scikit-learn matplotlib pandas nbformat

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))

## 1) Dataset choice

This lab supports two options:

### Option A (recommended): Local image folders
Prepare either:

**A1. Preferred structure**
data/
train/
class_1/
class_2/
val/ (optional)
test/ (optional)

**A2. Single-folder structure**
data/
class_1/
class_2/

The notebook will split automatically.

### Option B (fallback): CIFAR-10
Downloads on first use (internet needed).

In [ ]:
# --- CONFIG: the main knobs students should experiment with ---
# These parameters control BOTH learning behavior and memory usage.
#
# If Colab crashes ("RAM" warning):
# 1) Lower BATCH_SIZE (16 -> 8 -> 4)
# 2) Lower IMG_SIZE  (160 -> 128)
# 3) Keep CACHE_MODE='none' (or try 'disk' for speed without RAM usage)
#
# Memory note:
# - System RAM is mostly affected by caching + large batches/images.
# - GPU memory is affected by batch size, image size, and model size.

# =========================
# CONFIG (Colab-safe defaults)
# =========================

# Choose dataset mode
DATASET_MODE = 'cifar10'      # 'local' or 'cifar10'

# If using local folders, set the path (Colab examples):
# LOCAL_DATA_DIR = '/content/data'
# LOCAL_DATA_DIR = '/content/drive/MyDrive/amai/data'
LOCAL_DATA_DIR = 'data'

# Memory-safe defaults for Colab (you can increase later if stable):
IMG_SIZE = (160, 160)   # 224x224 is heavier; 160x160 works well for MobileNetV2
BATCH_SIZE = 16         # reduce if you still see memory errors

# Dataset caching:
# 'none'  = safest for RAM
# 'disk'  = faster than none, uses local disk (/tmp) instead of RAM
# 'memory'= fastest but can crash Colab for large datasets
CACHE_MODE = 'none'
CACHE_FILE = '/tmp/tfdata_cache'

# Optional GPU efficiency
MIXED_PRECISION = True  # helps speed + GPU memory (does not increase system RAM)

SEED = 42
EPOCHS_BASELINE = 8
EPOCHS_TRANSFER = 6
EPOCHS_FINETUNE = 4

tf.keras.utils.set_random_seed(SEED)

np.random.seed(SEED)
random.seed(SEED)


# Colab GPU setup (optional but recommended)
# Prevent TensorFlow from grabbing all GPU memory at once
_gpus = tf.config.list_physical_devices('GPU')
for _gpu in _gpus:
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except Exception as e:
        print('GPU memory growth not set:', e)

# Mixed precision can reduce GPU memory usage and speed up training on GPUs
if MIXED_PRECISION and _gpus:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision policy:', mixed_precision.global_policy())


In [ ]:
# --- Data loading ---
# This block builds train/validation/test datasets.
#
# What you should understand:
# - image_dataset_from_directory reads images from folders and creates (image, label) batches.
# - Labels come from folder names (each folder = one class).
# - We set shuffle=False for validation/test to keep evaluation deterministic.
#
# Experiment ideas (safe):
# - Try different IMG_SIZE/BATCH_SIZE in CONFIG.
# - If using your own data, verify the folder structure and class names.


AUTOTUNE = tf.data.AUTOTUNE

def show_batch(images, labels, class_names, n=12):
    plt.figure(figsize=(10, 7))
    n = min(n, images.shape[0])
    for i in range(n):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

def build_local_datasets():
    train_dir = os.path.join(LOCAL_DATA_DIR, "train")  # data/train/<class_name>/*.jpg
    val_dir   = os.path.join(LOCAL_DATA_DIR, "val")    # data/val/<class_name>/*.jpg (optional)
    test_dir  = os.path.join(LOCAL_DATA_DIR, "test")   # data/test/<class_name>/*.jpg (recommended)

    # Preferred structure: data/train exists
    if os.path.isdir(train_dir):
        # Training set: shuffled by default inside Keras utility
        train_ds = tf.keras.utils.image_dataset_from_directory(
            train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED
        )
        class_names = train_ds.class_names

        if os.path.isdir(val_dir):
            # Validation set: shuffle=False for stable metrics
            val_ds = tf.keras.utils.image_dataset_from_directory(
                val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=False
            )
        else:
            # Auto-split train into train/val (when you do not have a separate val/ folder)
            # Important: validation is NOT shuffled so metrics are stable.
            train_ds = tf.keras.utils.image_dataset_from_directory(
                train_dir, validation_split=0.2, subset="training",
                seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE
            )
            val_ds = tf.keras.utils.image_dataset_from_directory(
                train_dir, validation_split=0.2, subset="validation",
                seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
            )
            class_names = train_ds.class_names

        if os.path.isdir(test_dir):
            # Test set: shuffle=False so mistakes are reproducible
            test_ds = tf.keras.utils.image_dataset_from_directory(
                test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=False
            )
        else:
            # For real projects, create a proper test set.
            test_ds = val_ds

        return train_ds, val_ds, test_ds, class_names

    # Single-folder: data/<class_name>/...
    if os.path.isdir(LOCAL_DATA_DIR):
        # Single-folder mode: data/<class_name>/*.jpg
        full_ds = tf.keras.utils.image_dataset_from_directory(
            LOCAL_DATA_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED
        )
        class_names = full_ds.class_names

        ds_size = full_ds.cardinality().numpy()  # number of batches (approx), not raw image count
        train_size = int(0.70 * ds_size)
        val_size   = int(0.15 * ds_size)
        test_size  = ds_size - train_size - val_size

        full_ds = full_ds.shuffle(1000, seed=SEED, reshuffle_each_iteration=False)
        train_ds = full_ds.take(train_size)
        val_ds   = full_ds.skip(train_size).take(val_size)
        test_ds  = full_ds.skip(train_size + val_size).take(test_size)

        return train_ds, val_ds, test_ds, class_names

    raise FileNotFoundError(
        f"Dataset not found at '{LOCAL_DATA_DIR}'. Create 'data/' or update LOCAL_DATA_DIR."
    )

def build_cifar10_datasets():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

    n_val = int(len(x_train) * 0.2)
    x_val, y_val = x_train[:n_val], y_train[:n_val]
    x_tr, y_tr   = x_train[n_val:], y_train[n_val:]

    def preprocess(x, y):  # resize + shape labels for CIFAR-10
        x = tf.image.resize(x, IMG_SIZE)
        return x, tf.squeeze(y)

    train_ds = tf.data.Dataset.from_tensor_slices((x_tr, y_tr)).shuffle(5000, seed=SEED).map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    val_ds   = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    test_ds  = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)

    class_names = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
    return train_ds, val_ds, test_ds, class_names

if DATASET_MODE == "local":
    train_ds, val_ds, test_ds, class_names = build_local_datasets()
elif DATASET_MODE == "cifar10":
    train_ds, val_ds, test_ds, class_names = build_cifar10_datasets()
else:
    raise ValueError("DATASET_MODE must be 'local' or 'cifar10'")

print("Classes:", class_names)
print("Number of classes:", len(class_names))


Loading CIFAR-10 via TensorFlow Datasets (optimized CDN)...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

KeyboardInterrupt: 

In [ ]:
# --- tf.data optimization (speed + memory discipline) ---
# We keep the pipeline efficient using:
# - shuffle(): only for training, improves learning
# - cache(): OPTIONAL. 'memory' is fast but can crash Colab RAM; prefer 'none' or 'disk'.
# - prefetch(AUTOTUNE): overlaps data loading with model computation.
#
# If you want to experiment with speed:
# - Try CACHE_MODE='disk' (usually safe in Colab).

def optimize(ds, training=False, cache_suffix=''):
    # Memory-safe tf.data optimization.
    # IMPORTANT: caching in RAM can crash Colab for large datasets.
    if training:
        ds = ds.shuffle(1000, seed=SEED, reshuffle_each_iteration=True)

    if CACHE_MODE == 'memory':
        ds = ds.cache()
    elif CACHE_MODE == 'disk':
        # Use different cache files for train/val/test to avoid collisions
        ds = ds.cache(CACHE_FILE + cache_suffix)
    # else: no cache

    return ds.prefetch(AUTOTUNE)

train_ds_opt = optimize(train_ds, training=True, cache_suffix='_train')
val_ds_opt   = optimize(val_ds, training=False, cache_suffix='_val')
test_ds_opt  = optimize(test_ds, training=False, cache_suffix='_test')

for images, labels in train_ds.take(1):
    print('Batch:', images.shape, labels.shape, images.dtype)
    show_batch(images, labels, class_names, n=12)


## 2) Normalization and augmentation
Augmentation should represent **realistic** variation (e.g., small rotations, mild zoom, horizontal flip where appropriate).

In [ ]:
# --- Input processing ---
# 1) Normalization rescales pixel values to [0,1]. This stabilizes optimization.
# 2) Data augmentation creates realistic variations to improve generalization.
#    Note: augmentation is applied during training (fit), not during evaluation.
#
# Experiment ideas:
# - Increase/decrease rotation/zoom.
# - Add RandomContrast or RandomBrightness (but keep it realistic).


normalization = layers.Rescaling(1./255)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
], name="augmentation")

## 3) Baseline CNN
The baseline answers an important research question:
> “Is the problem learnable with a simple model before using a large pre-trained network?”

In [ ]:
# --- Baseline CNN (learn features from scratch) ---
# Conceptual map:
# - Conv2D layers learn local patterns (edges, textures).
# - MaxPooling reduces spatial size and keeps the strongest signals.
# - Deeper conv layers learn higher-level patterns.
# - GlobalAveragePooling converts feature maps into a compact vector.
# - Dropout reduces overfitting by randomly dropping features during training.
# - Dense softmax outputs class probabilities.
#
# Safe experiments (usually fine on Colab):
# - Change filter counts: 32/64/128 -> 16/32/64 (lighter) or 64/128/256 (heavier)
# - Add BatchNormalization after Conv2D
# - Change Dropout(0.30) -> 0.1 or 0.5 and observe curves


def make_baseline_cnn(num_classes: int) -> keras.Model:
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = normalization(inputs)
    x = data_augmentation(x)

    x = layers.Conv2D(32, 3, activation="relu")(x)  # small filters detect simple patterns
    x = layers.MaxPooling2D()(x)  # downsample: fewer pixels, more invariance
    x = layers.Conv2D(64, 3, activation="relu")(x)  # more filters = richer features
    x = layers.MaxPooling2D()(x)  # downsample: fewer pixels, more invariance
    x = layers.Conv2D(128, 3, activation="relu")(x) # deeper features: parts/objects

    x = layers.GlobalAveragePooling2D()(x)  # compress spatial maps -> vector
    x = layers.Dropout(0.30)(x)  # regularization: helps generalization
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    return keras.Model(inputs, outputs, name="baseline_cnn")

baseline_model = make_baseline_cnn(len(class_names))
baseline_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),  # learning rate (try 3e-4 or 1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
baseline_model.summary()


In [ ]:
# --- Training loop ---
# EarlyStopping watches validation loss and stops when it stops improving.
# This prevents wasted computation and often reduces overfitting.
#
# If training is slow or unstable:
# - Lower EPOCHS_* (CONFIG)
# - Lower learning rate (Adam(1e-3) -> 3e-4)
# - Lower IMG_SIZE or BATCH_SIZE (CONFIG) if you see memory errors


callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
]

history_base = baseline_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=EPOCHS_BASELINE,
    callbacks=callbacks
)

In [ ]:
# --- Learning curves ---
# These plots are how we diagnose model behavior:
# - If train improves but val gets worse -> overfitting
# - If both are poor -> underfitting or data/label issues
#
# Good practice: after each experiment, compare curves + confusion matrix, not only accuracy.


def plot_history(history, title):
    h = history.history
    epochs = range(1, len(h["loss"]) + 1)

    plt.figure(figsize=(10,4))
    plt.plot(epochs, h["loss"], label="train loss")
    plt.plot(epochs, h["val_loss"], label="val loss")
    plt.title(f"{title} — Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(epochs, h["accuracy"], label="train acc")
    plt.plot(epochs, h["val_accuracy"], label="val acc")
    plt.title(f"{title} — Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.show()

plot_history(history_base, "Baseline CNN")

## 4) Evaluation: confusion matrix + error analysis
A strong evaluation is not only an overall score.  
You should **locate systematic error patterns** and support them with examples.

In [ ]:
# --- Evaluation (streaming: Colab RAM friendly) ---
# Why streaming?
# - Storing the full test set (all images) can exceed 12 GB RAM.
# - We store only labels/predictions (small) + a few misclassified examples.
#
# What to learn here:
# - Confusion matrix: shows which classes are confused with which.
# - Classification report: precision/recall/F1 per class.
# - Error analysis: look at representative mistakes and hypothesize causes.
#
# If you still hit memory issues during evaluation:
# - Set max_misclassified=0 (stores no images)
# - Lower BATCH_SIZE (CONFIG)

def predict_labels_stream(model, ds, max_misclassified=12):
    """
    Streaming predictions that are safe for Colab RAM.
    Returns: y_true, y_pred, misclassified_samples
    misclassified_samples is a small list of (image, true_label, pred_label).
    """
    y_true, y_pred = [], []
    mis = []

    for x, y in ds:
        # forward pass without storing batch outputs in large arrays
        probs = model(x, training=False)
        preds = tf.argmax(probs, axis=1)

        y_np = y.numpy().astype(int)
        p_np = preds.numpy().astype(int)

        y_true.extend(y_np.tolist())
        y_pred.extend(p_np.tolist())

        if max_misclassified > 0 and len(mis) < max_misclassified:
            mismatch = (p_np != y_np)
            if mismatch.any():
                # store only a few misclassified images, not the full test set
                x_np = x.numpy()
                # ensure displayable uint8 images
                if x_np.dtype != np.uint8:
                    x_np = np.clip(x_np, 0, 255).astype('uint8')
                for img, yt, yp, m in zip(x_np, y_np, p_np, mismatch):
                    if m:
                        mis.append((img, int(yt), int(yp)))
                        if len(mis) >= max_misclassified:
                            break

    return np.array(y_true), np.array(y_pred), mis

def plot_confusion(cm, names, title):
    plt.figure(figsize=(7,6))
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(names))
    plt.xticks(ticks, names, rotation=45, ha='right')
    plt.yticks(ticks, names)
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout(); plt.show()

def show_misclassified_samples(mis, names):
    if not mis:
        print('No misclassified examples found (or max_misclassified=0).')
        return
    n = len(mis)
    plt.figure(figsize=(10,7))
    for i, (img, yt, yp) in enumerate(mis[:12]):
        ax = plt.subplot(3,4,i+1)
        plt.imshow(img)
        plt.title(f'T:{names[yt]}\nP:{names[yp]}')
        plt.axis('off')
    plt.tight_layout(); plt.show()

# --- Baseline evaluation (streaming) ---
y_true, y_pred, mis_base = predict_labels_stream(baseline_model, test_ds_opt, max_misclassified=12)

cm = confusion_matrix(y_true, y_pred)
plot_confusion(cm, class_names, 'Baseline CNN — Confusion Matrix')

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
show_misclassified_samples(mis_base, class_names)


## 5) Transfer learning (MobileNetV2)
Transfer learning is academically justified when you want strong feature representations with limited data.

In [ ]:
# --- Transfer learning (pre-trained CNN as a feature extractor) ---
# MobileNetV2 is trained on ImageNet, so it already learned general visual features.
# We freeze the backbone and train a small new 'head' for our classes.
#
# Key concepts:
# - preprocess_input: matches the input scale expected by MobileNetV2
# - frozen backbone: faster training and less overfitting on small datasets
#
# Safe experiments:
# - Try different Dropout values in the head
# - Try a smaller IMG_SIZE (128) if you get memory errors
# - (Advanced) swap backbone to EfficientNetB0 (heavier; may require smaller batch)


def make_transfer_model(num_classes):
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet"
    )
    backbone.trainable = False  # freeze: do NOT update pretrained weights

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)  # required scaling for this backbone
    x = data_augmentation(x)
    x = backbone(x, training=False)  # run feature extractor
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = keras.Model(inputs, outputs, name="transfer_mobilenetv2")
    return model, backbone

transfer_model, backbone = make_transfer_model(len(class_names))
transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
transfer_model.summary()


In [ ]:
# --- Train transfer model (frozen backbone) ---
# This stage usually converges quickly because only the new head is learning.
# If validation improves fast, that's typical of transfer learning.


history_tl = transfer_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=EPOCHS_TRANSFER,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
)
plot_history(history_tl, "Transfer Learning (Frozen Backbone)")

In [ ]:
# --- Evaluate transfer model ---
# Keep evaluation identical across models so comparisons are fair.

# --- Transfer learning evaluation (streaming) ---
y_true_tl, y_pred_tl, mis_tl = predict_labels_stream(transfer_model, test_ds_opt, max_misclassified=12)

cm_tl = confusion_matrix(y_true_tl, y_pred_tl)
plot_confusion(cm_tl, class_names, 'Transfer Learning — Confusion Matrix')

print(classification_report(y_true_tl, y_pred_tl, target_names=class_names, digits=3))
show_misclassified_samples(mis_tl, class_names)


## 6) Optional: fine-tuning (advanced)
Fine-tuning can improve performance, but it should be done with:
- a **small learning rate**
- careful validation monitoring

In [ ]:
# --- Optional fine-tuning (advanced) ---
# Fine-tuning unfreezes some of the backbone so it can adapt to your dataset.
# IMPORTANT:
# - Use a SMALL learning rate (1e-5 is a common starting point)
# - Unfreeze only the last N layers first
#
# If you see instability or memory errors:
# - Lower BATCH_SIZE (CONFIG)
# - Reduce N_UNFREEZE
# - Turn off MIXED_PRECISION if your GPU has issues (CONFIG)


# OPTIONAL: fine-tune last layers
backbone.trainable = True

N_UNFREEZE = 40  # try 20 for safer fine-tuning, or 80 for stronger adaptation
for layer in backbone.layers[:-N_UNFREEZE]:
    layer.trainable = False

transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = transfer_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=EPOCHS_FINETUNE,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)]
)

plot_history(history_ft, "Fine-tuning (Last Layers Unfrozen)")


In [ ]:
# --- Evaluate fine-tuned model ---
# Check if fine-tuning helped AND whether it increased confusion for some classes.
# Sometimes fine-tuning improves one class but harms another; the confusion matrix reveals this.

# Evaluate fine-tuned model (streaming)
y_true_ft, y_pred_ft, mis_ft = predict_labels_stream(transfer_model, test_ds_opt, max_misclassified=12)

cm_ft = confusion_matrix(y_true_ft, y_pred_ft)
plot_confusion(cm_ft, class_names, 'Fine-tuned Transfer — Confusion Matrix')

print(classification_report(y_true_ft, y_pred_ft, target_names=class_names, digits=3))
show_misclassified_samples(mis_ft, class_names)


## 7) Results summary table (recommended)
A results table supports academic clarity and fair comparison.

In [ ]:
# --- Results table ---
# We summarize multiple metrics because accuracy alone can be misleading.
# - macro_f1 treats each class equally (good for imbalanced datasets)
# - weighted_f1 weights by class frequency (reflects average performance on the dataset)


def to_row(name, y_true, y_pred):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "model": name,
        "accuracy": rep["accuracy"],
        "macro_f1": rep["macro avg"]["f1-score"],
        "weighted_f1": rep["weighted avg"]["f1-score"],
    }

rows = []
rows.append(to_row("Baseline CNN", y_true, y_pred))
rows.append(to_row("Transfer (frozen)", y_true_tl, y_pred_tl))
rows.append(to_row("Transfer (fine-tuned)", y_true_ft, y_pred_ft))

df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
df

In [ ]:
# --- Export results ---
# Saving a CSV is useful for reports and for tracking experiments over time.


df.to_csv("lab2_results_summary.csv", index=False)
print("Saved: lab2_results_summary.csv")

## 8) Short memo (submit inside the notebook)
Write **8–12 lines** using academically clear language:

1. **Goal**: What is the task and why does it matter?  
2. **Data**: classes, approximate size, and any quality risks.  
3. **Methods**: baseline vs transfer learning (and fine-tuning if used).  
4. **Evaluation**: which metrics and why.  
5. **Error analysis**: 1–2 typical confusions with example images.  
6. **Recommendation**: which model you would deploy and why.  
7. **Responsible AI**: one risk and one mitigation.

### Memo (student write-up)
- **Goal:**  
- **Data:**  
- **Methods:**  
- **Evaluation:**  
- **Error analysis:**  
- **Recommendation:**  
- **Responsible AI risk + mitigation:**

## Submission checklist
- [ ] Baseline CNN trained + evaluated  
- [ ] Transfer learning trained + evaluated  
- [ ] Learning curves shown  
- [ ] Confusion matrix + ≥10 misclassified examples shown  
- [ ] Memo completed in an academically structured form  
- [ ] Notebook runs top-to-bottom without errors